# EveryCli — fine-tune the semantic model

Fine-tunes `paraphrase-multilingual-MiniLM-L12-v2` on EveryCli's own corpus
(`training/pairs.jsonl`, built by `training/build_pairs.py`) using
`MultipleNegativesRankingLoss` — it only needs positive pairs (different
phrasings of the same intent), in-batch examples act as negatives.

Runs identically on Google Colab's free T4 GPU or locally if you have one.
The base model is small (~118M params), so a few hundred/thousand short-text
pairs train in minutes, not hours.

**Do not add `eval/confusion_set.yaml` examples to the training pairs** —
`build_pairs.py` already excludes them; that file stays the untouched
non-regression gate checked in the last cell and in `tools/evaluate_confusion.py`.

## 1. Setup

On Colab: clone the repo (or upload it) and mount Drive for checkpoints, since
free-tier sessions can disconnect after ~90 minutes idle.

In [ ]:
# Colab only — skip these two cells when running locally with the repo already checked out.
!pip install -q sentence-transformers
!git clone https://github.com/HE11032006/EveryCli.git
%cd EveryCli

In [ ]:
# Colab only — mount Drive so checkpoints survive a disconnect.
from google.colab import drive
drive.mount('/content/drive')
CHECKPOINT_DIR = '/content/drive/MyDrive/everycli-finetune-checkpoints'

In [ ]:
# Local run — comment out the Colab cells above and uncomment this instead.
# CHECKPOINT_DIR = 'training/output/checkpoints'

## 2. Build (or reuse) the training pairs

`training/pairs.jsonl` is already checked into the repo (generated from the
current corpus, with every `eval/confusion_set.yaml` scenario excluded).
Regenerate it here only if the corpus changed since it was last committed.

In [ ]:
!python training/build_pairs.py

In [ ]:
import json
from pathlib import Path
from sentence_transformers import InputExample

pairs_path = Path('training/pairs.jsonl')
examples = []
for line in pairs_path.read_text(encoding='utf-8').splitlines():
    row = json.loads(line)
    examples.append(InputExample(texts=[row['text_a'], row['text_b']]))

print(f"{len(examples)} training pairs loaded")

## 3. Fine-tune

In [ ]:
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, losses

BASE_MODEL = 'paraphrase-multilingual-MiniLM-L12-v2'
OUTPUT_DIR = 'training/output/everycli-minilm-ft'
EPOCHS = 4
BATCH_SIZE = 32

model = SentenceTransformer(BASE_MODEL)
train_dataloader = DataLoader(examples, shuffle=True, batch_size=BATCH_SIZE)
train_loss = losses.MultipleNegativesRankingLoss(model)

warmup_steps = int(len(train_dataloader) * EPOCHS * 0.1)

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=EPOCHS,
    warmup_steps=warmup_steps,
    checkpoint_path=CHECKPOINT_DIR,
    checkpoint_save_steps=200,
    show_progress_bar=True,
)

model.save(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}")

## 4. Non-regression gate — compare against the current baseline

Do not adopt the fine-tuned model unless this matches or beats the baseline
captured before starting (run `python tools/evaluate_confusion.py` once
*without* `EVERYCLI_MODEL_PATH` beforehand, on the same corpus, to get that
number). `EVERYCLI_MODEL_PATH` overrides the model used by
`SemanticMatcher._load_model` for exactly this kind of comparison — see
`everycli/infra/semantic_matcher.py`.

In [ ]:
import os
os.environ['EVERYCLI_MODEL_PATH'] = 'training/output/everycli-minilm-ft'
!python tools/evaluate_confusion.py --show-top1-misses

## 5. Adopt (only if step 4 didn't regress)

Either:
- push `training/output/everycli-minilm-ft` to a Hugging Face Hub repo and
  point `MODEL_NAME` in `everycli/infra/semantic_matcher.py` at it, or
- copy it into the `models/` folder the frozen PyInstaller build already
  looks for (`sys._MEIPASS / "models" / model_name`) and swap the
  "Download Model" step in `.github/workflows/build.yml` to use this
  artifact instead of downloading the base model.